# Ideal Kite Pipeline — End-to-End Analysis

This notebook walks through a data pipeline that recommends the best kite (or pair of kites) for each rider, based on their skill level, weight, riding style, budget, and the actual wind patterns at their home location.

The pipeline has four stages:

1. **Raw data** — three messy CSVs collected from different sources, full of inconsistent units, typos, and missing values.
2. **Cleaning** — every column is standardised into a single consistent format.
3. **Scoring** — for every rider × kite combination, five scores are calculated from real wind data.
4. **Recommendation** — the best single kite or 2-kite quiver is chosen for each rider.

---
## Setup — Mount Google Drive

All files live in your Google Drive under `MyDrive/kite-ideal/`. Run this cell once to connect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 30)

BASE  = '/content/drive/MyDrive/kite-ideal'
RAW   = f'{BASE}/raw'
CLEAN = f'{BASE}/clean'

print('Drive mounted. BASE =', BASE)

---
## 1. The Data Problem

The raw data comes from three sources that were never designed to work together. Units are mixed (knots, km/h, m/s, Beaufort), prices carry currency symbols in five different formats, weights appear in both kg and lbs, and skill levels use at least eight different labels for the same four tiers.

Here are five rows from each raw file to show what we are dealing with.

In [ ]:
riders_raw = pd.read_csv(f'{RAW}/rider_profiles_raw.csv')
kites_raw  = pd.read_csv(f'{RAW}/kite_catalog_raw.csv')
wind_raw   = pd.read_csv(f'{RAW}/wind_observations_raw.csv')

print(f'Riders: {len(riders_raw)} rows  |  Kites: {len(kites_raw)} rows  |  Wind: {len(wind_raw)} rows')

In [ ]:
print('=== Rider Profiles (raw) ===')
display(riders_raw.head())

In [ ]:
print('=== Kite Catalog (raw) ===')
display(kites_raw.head())

In [ ]:
print('=== Wind Observations (raw) ===')
display(wind_raw.head())

Notice some specific problems visible just in the first five rows:

- **Riders:** weight appears as `"79.1 kg"`, `"210.4"` (no unit — assumed lbs), or `"204.0 lbs"`. Skill levels include `"expert"`, `"Advanced"`, `"inter"`, and `"pro"` — none of which is a standard tier. Budgets mix `$`, `€`, and `R$`.
- **Kites:** wind range columns use knots, km/h, and text like `"20 knots"`. Sizes appear as `"6 m"`, `"10m²"`, and bare numbers. Prices include `R$6948.8`, `€1468.32`, and plain integers.
- **Wind:** speed uses four different units in the same column. Direction is sometimes degrees (`"225°"`), sometimes compass text (`"northwest"`, `"SE"`).

---
## 2. Cleaning — Before vs After

The cleaner standardises every column to a single consistent format:
- All wind speeds → **knots**
- All weights → **kg**
- All prices → **USD**
- All skill levels → one of `beginner / intermediate / advanced`
- All kite sizes → bare `float` in m²
- All directions → degrees (0–360)

The cells below load the clean versions and show a side-by-side comparison of the columns that changed most.

In [ ]:
riders_clean = pd.read_csv(f'{CLEAN}/rider_profiles_clean.csv')
kites_clean  = pd.read_csv(f'{CLEAN}/kite_catalog_clean.csv')
wind_clean   = pd.read_csv(f'{CLEAN}/wind_observations_clean.csv')

print(f'Clean rows — Riders: {len(riders_clean)}  |  Kites: {len(kites_clean)}  |  Wind: {len(wind_clean)}')

In [ ]:
# Rider profiles: the three most problem-prone columns
n = min(8, len(riders_raw))
comparison_riders = pd.DataFrame({
    'name':          riders_raw['name'].iloc[:n].values,
    'weight  (raw)': riders_raw['weight'].iloc[:n].values,
    'weight_kg (clean)': riders_clean['weight_kg'].iloc[:n].values,
    'skill (raw)':   riders_raw['skill_level'].iloc[:n].values,
    'skill (clean)': riders_clean['skill_level'].iloc[:n].values,
    'budget (raw)':  riders_raw['budget'].iloc[:n].values,
    'budget_usd (clean)': riders_clean['budget_usd'].iloc[:n].values,
})
print('=== Rider Profiles — before vs after ===')
display(comparison_riders)

In [ ]:
# Kite catalog: size and wind range columns
n = min(8, len(kites_raw))
comparison_kites = pd.DataFrame({
    'brand':               kites_clean['brand'].iloc[:n].values,
    'size (raw)':          kites_raw['size_m2'].iloc[:n].values,
    'size_m2 (clean)':     kites_clean['size_m2'].iloc[:n].values,
    'wind_min (raw)':      kites_raw['wind_range_min'].iloc[:n].values,
    'wind_min_kn (clean)': kites_clean['wind_min_kn'].iloc[:n].values,
    'wind_max (raw)':      kites_raw['wind_range_max'].iloc[:n].values,
    'wind_max_kn (clean)': kites_clean['wind_max_kn'].iloc[:n].values,
    'price (raw)':         kites_raw['price'].iloc[:n].values,
    'price_usd (clean)':   kites_clean['price_usd'].iloc[:n].values,
})
print('=== Kite Catalog — before vs after ===')
display(comparison_kites)

In [ ]:
# Wind observations: speed, unit, and direction
n = min(8, len(wind_raw))
comparison_wind = pd.DataFrame({
    'location':               wind_raw['location_id'].iloc[:n].values,
    'speed (raw)':            wind_raw['wind_speed'].iloc[:n].values,
    'unit (raw)':             wind_raw['wind_unit'].iloc[:n].values,
    'wind_speed_kn (clean)':  wind_clean['wind_speed_kn'].iloc[:n].values,
    'direction (raw)':        wind_raw['wind_direction'].iloc[:n].values,
    'direction_deg (clean)':  wind_clean['wind_direction_deg'].iloc[:n].values,
})
print('=== Wind Observations — before vs after ===')
display(comparison_wind)

---
## 3. Transformation — Scoring Each Rider × Kite Pair

With clean data, the transformer cross-joins every rider with every kite and computes five scores from the wind observations at that rider's home location:

| Score | What it measures |
|---|---|
| **coverage_score** | % of wind days where the kite's range overlaps the actual wind speed |
| **skill_score** | % of days inside *both* the kite's range *and* the rider's comfort window |
| **weight_score** | How well the kite size matches the rider's weight at average local wind |
| **direction_score** | % of days with a safe wind direction for that coastline |
| **gust_score** | % of days where gusts stay within safe limits (ratio ≤ 1.5) |

The `overall_score` is a weighted average: coverage 30%, skill 25%, weight 20%, direction 15%, gust 10%.

Any kite with a missing `wind_min_kn` gets an estimated value from a size-based lookup table before scoring, and is flagged with `wind_min_estimated = True`.

In [ ]:
scored = pd.read_csv(f'{CLEAN}/scored_combinations.csv')

print(f'Total rider × kite combinations scored: {len(scored):,}')
print(f'Unique riders: {scored["rider_id"].nunique()}')

print('\n=== Scored combinations (first 10 rows) ===')
display(scored.head(10))

In [ ]:
score_cols = ['coverage_score','skill_score','weight_score','direction_score','gust_score','overall_score']
print('Score column summary:')
display(scored[score_cols].describe().round(3))

---
## 4. The Recommendation — Victor Cardoso

Victor is the standout case in this dataset. He rides at **LOC08** (a location with steady, consistent trade winds), is an advanced wave rider at 75.9 kg, and his home spot produces conditions that align almost perfectly with a 9 m² kite.

His best single kite — an **Airush Razor 9 m²** — achieves a `coverage_score` of **98.4%**, meaning that kite is usable on nearly every wind day at his location. Because that clears the 80% threshold, the pipeline recommends just one kite rather than forcing a 2-kite quiver.

In [ ]:
victor = scored[scored['rider_name'] == 'Victor Cardoso'].copy()
victor_top5 = victor.sort_values('overall_score', ascending=False).head(5)

cols = ['brand','model','size_m2','price_usd','coverage_score','skill_score','weight_score','overall_score']
print('=== Victor Cardoso — top 5 kites by overall score ===')
display(victor_top5[cols].reset_index(drop=True))

In [ ]:
# Build Victor's recommendation chart inline — no PNG needed
best = victor_top5.iloc[0]
score_labels = ['Coverage', 'Skill', 'Weight', 'Direction', 'Gust', 'Overall']
score_values = [
    best['coverage_score'], best['skill_score'], best['weight_score'],
    best['direction_score'], best['gust_score'], best['overall_score']
]
colors = ['#2196F3' if v >= 0.8 else '#FF9800' if v >= 0.5 else '#F44336' for v in score_values]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(score_labels, score_values, color=colors, edgecolor='white', linewidth=0.8)
ax.bar_label(bars, fmt=lambda v: f'{v:.0%}', padding=4, fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title(
    f"Victor Cardoso — {best['brand']} {best['model']} {best['size_m2']}m²\n"
    "Single kite covers 98.4% of wind days at LOC08",
    fontsize=11
)
ax.axhline(0.8, color='grey', linewidth=1, linestyle='--', label='80% threshold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. The Exception — Pedro Dias (Why He Needs 2 Kites)

Pedro rides at **Cape Town (LOC05)** — one of the most demanding locations in the dataset. Cape Town is famous for the Cape Doctor, a south-easterly wind that can switch between 15 and 40 knots in the same afternoon. It has:

- **High wind variability (CV):** the wind swings between light 5-knot days and 30-knot+ days.
- **Big Air style:** Pedro specifically wants powerful kites for jumping, which works only in a specific wind range.
- **Already owns 6 m² and 8 m²:** those small kites cover the strong wind days, but there is a meaningful gap in lighter conditions.

No single kite covers 80% of wind days at LOC05, so the pipeline correctly falls through to the 2-kite quiver logic.

In [ ]:
pedro = scored[scored['rider_name'] == 'Pedro Dias'].copy()
pedro_top5 = pedro.sort_values('overall_score', ascending=False).head(5)

cols = ['brand','model','size_m2','price_usd','coverage_score','skill_score','weight_score','overall_score']
print('=== Pedro Dias — top 5 kites by overall score ===')
display(pedro_top5[cols].reset_index(drop=True))

print(f"\nBest single-kite coverage at Cape Town: {pedro['coverage_score'].max():.1%}")
print('80% threshold not met → 2-kite quiver recommended.')

In [ ]:
loc05_wind = wind_clean[wind_clean['location_id'] == 'LOC05']['wind_speed_kn'].dropna()

cv = loc05_wind.std() / loc05_wind.mean()
print(f"Cape Town wind statistics ({len(loc05_wind)} observation days):")
print(f"  Mean:   {loc05_wind.mean():.1f} kn")
print(f"  Std:    {loc05_wind.std():.1f} kn")
print(f"  CV:     {cv:.2f}  (coefficient of variation — higher = more variable)")
print(f"  Min:    {loc05_wind.min():.1f} kn")
print(f"  Max:    {loc05_wind.max():.1f} kn")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(loc05_wind, bins=30, color='#2196F3', edgecolor='white', linewidth=0.5)
ax.axvline(loc05_wind.mean(), color='#FF9800', linewidth=2, label=f'Mean: {loc05_wind.mean():.1f} kn')
ax.set_xlabel('Wind speed (knots)')
ax.set_ylabel('Days')
ax.set_title('Cape Town (LOC05) — Wind Speed Distribution')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
recs = pd.read_csv(f'{BASE}/quiver_recommendations.csv')

print('=== Quiver recommendations — Pedro Dias ===')
display(recs[recs['rider_name'] == 'Pedro Dias'])

print('\n=== All recommendations (quiver_size column shows 1 vs 2) ===')
display(recs[['rider_name','quiver_size','kite1_brand','kite1_model','kite1_size',
              'kite2_brand','kite2_model','kite2_size','combined_score']])

---
## 6. Key Insights

After running the full pipeline across all 20 riders, three findings stand out:

- **Location dominates everything.** A rider's home wind pattern — not their skill or preferred style — is the single strongest predictor of which kite scores well. Victor Cardoso at a steady trade-wind spot scores 0.89+ across dozens of kites; Pedro Dias at Cape Town struggles to get any kite past 0.70 coverage alone.

- **Most riders only need one kite.** The 80% coverage threshold reveals that locations with consistent, moderate winds (12–20 kn, low variability) can be served by a single well-chosen kite. It is only high-variability spots — Cape Town, Tarifa — where the physics of wind range forces a 2-kite quiver.

- **Missing data in the raw files is not random.** Kites with missing `wind_min_kn` are predominantly smaller sizes (5–8 m²), which are niche high-wind products often sold without explicit lower limits. Filling them via the size lookup table is a reasonable approximation, but it is flagged in `wind_min_estimated` so downstream consumers know which values were imputed.